## Flash attention

Source: [Priyam](https://www.youtube.com/watch?v=CwLelq5SX7g)

#### What's wrong with normal softmax attention?

- we always need to materialize the full $O(T^2)$ matrix which is very expensive at long context lengths, where $T$ = max_seq_len
- consider the size of the Query matrix
  - sequence length x embedding dimension: $O(Td)$
- now consider the size of the $QK^T$ matrix (i.e. the query-key matrix)
  - sequence length x sequence length $O(T^2)$

For max sequence length $T$ = 128k, embed dim $d$ = 8192

- 128k x 128k is an order of magnitude larger than 128k x 8192

So we really want to shrink how much of this $QK^T$ matrix we materalize! Especially since frontier models now have >1M context length 🤯

#### How do we do this?

So we might deduce that the first row of the attention output is actually just

- First row of the query matrix $Q$, dotted with the first row of the K matrix (i.e. the first query token ($Q_1$), 'attends' to the first key token ($K$))
- Then that output forms the first element of our $QK^T$ matrix!
- But...we can't do softmax with just this one value, we need the max and the exponentiated sum of the entire $QK^T$ row (or a row of $A$ as named in the diagram below)

Recall: $$ softmax(x*{i,j}) = \dfrac{e^{x*{i,j} - max_i}}{\sum_a^T e^{x_a - max_i}}$$

_where $i$ denotes the values in the row, and $i,j$ denotes the j'th value in the i'th row_

<img src="../assets/flash-1.png" width="500">

Source: [Priyam](https://www.youtube.com/watch?v=CwLelq5SX7g)

_Why not just do the first query row dotted with all the columns in $K^T$?_

- This still requires us to materialize K - $O(Td)$ which we cannot load into SRAM (on-chip shared memory)
- We want to fit all our calculations onto a 100-200kb chip (SRAM)

#### Online softmax

We'll apply the same philosophy as above, but instead store the max and the exponentiated sum (denominator of softmax) for each row - once we have that, we can find each element of the final attention matrix easily

- Find a column of $Q$ dotted with a column of $K$ - save the value, add to a saved `sum` tensor representing our denominator in softmax, and a `max` tensor, representing our max value in the exponent in the numerator.

- We'll iteratively update these values. Until we get the true value once we have computed an entire row of $QK^T$

- Thus, never needing to materialize the full $QK^T$ or $K, V$ matrices! With this method we'll only ever need a row and column from $Q, K$ to get our softmax output

- Then at the end, we'll use these values to iteratively compute the attention output matrix one vector at a time, never materializing the intermediate $QK^T$ matrix

_NOTE: This is fused attention, meaning in one go, we can take one $Q$ and $K$ vector, the row `sum` and `max` and compute the corresponding final attention output. One operation on SRAM, not memory bandwidth bound!_

Here's the brief formula below where $m_i$ represents the max of row i, and $d_i$ represents the exponentiated sum of row i. See that both values are running, in the sense that they get updated as we matmul $Q$ with its corresponding $K^T$ vectors:

<img src="../assets/flash-2.png" width="300">

Source: [Priyam](https://www.youtube.com/watch?v=CwLelq5SX7g)


In [ ]:
import torch
import math
import time

# input is the QK^T matrix: (T, T)
def softmax(x):

	row_max = torch.max(x, axis=1, keepdims=True).values # max value for every token
	x_stable = x-row_max 

	exp_x = torch.exp(x_stable)
	sum_x = torch.sum(exp_x, dim=1, keepdims=True) # sum along every row, getting the summed logits

	return exp_x/sum_x


# input is (T, d)
def online_softmax(x):

	# the trick of online softmax is that we compute the max and the exponentiated sum for every row
	# this allows us to chunk of the QK^T matrix, never materializing the full O(T^2) tensor

	M, N = x.shape # materialize a part of the T,T matrix 
	
	m_store = torch.zeros((M,)) # store the max for each row
	d_store = torch.zeros((M,)) # store the exponentiated sum for each row

	for i in range(M):

		xi = x[i] # taking the i'th token in x 

		m = float('-inf')
		d = 0 

		for j in range(N):

			xj = xi[j] # taking the j'th token, that token i would be attending to
			
			m_old = m
			m = max(m, xj) # set m to the maximum of these 2 numbers 
			
			# running exponentiated sum
			d = math.exp(xj-m) + d * math.exp(m_old-m) # d_{i+1} = d_i * exp(m_i - m_{t+1}) + exp(x_i - m_{i+1}) - update the sum with the new max, then add the new exponentiated value
		
		m_store[i] = m
		d_store[i] = d

	return m_store, d_store

def online_softmax_with_logsumexp(x):
	
	M, N = x.shape

	# logsumexp trick is basically that softmax(x) = x - (max + log(sum of all exponentiated terms))
	L_store = torch.zeros((M,))

	for i in range(M):

		xi = x[i]
		m = float('-inf')
		d = 0

		for j in range(N):

			xj = xi[j]

			m_old = m
			m = max(xj, m)
			d = d * math.exp(m_old-m) + math.exp(xj-m)

		L_store[i] = m + math.log(d)

	return L_store

def online_softmax_chunked(x, block_size=64):
	
	M, N = x.shape # differs for decode, chunking like in flash attention, cross-attention

	num_blocks = math.ceil(N / block_size)

	L_store = torch.zeros((M,))

	for i in range(M):

		xi = x[i]
		
		m = float('-inf')
		d = 0

		for j in range(num_blocks):

			block_start = j * block_size
			block_end = min(block_start + block_size, N) # clamping
			block = xi[block_start:block_end] # processes the tokens in the key matrix in chunks!

			block_max = block.max() # max of this chunk plz!
			m_old = m 
			m = max(m, block_max) # compare with prev block max

			d = d * math.exp(m_old - m) + torch.exp(block - m).sum().item() # sum of the previous blocks + new sum of this block!

		L_store[i] = m + torch.log(d) # add the block values to the block store! 

x = torch.randn(1024, 1024)
t0 = time.perf_counter()
online_softmax_with_logsumexp(x)
t1 = time.perf_counter()-t0
t1 # woah this is hella slow

1.9970431672409177

## Flash attention forward pass

- we're chunking up the query vectors, and that chunk will attend to a chunk of the key, value vectors!
- basically we fill up the final attention matrix a 'tile' at a time

- you might imagine the first row of the attention matrix (denoted by `output`) to be a representation of our token 1. It is a combination of other tokens, including itself, (from in our value matrix) weighted by a % of how much each token is relevant to it (given by our QK matrix)

- thus we can iteratively add to this representation, we might first add token 1,2,..,64's relationship with our first token, then add 65,..,128's

- over time we get in that first row of the attention matrix, how each other token relates to it, by iteratively adding!

<img src="../assets/flash-3.png" width="500">

Source: [Priyam](https://www.youtube.com/watch?v=CwLelq5SX7g)

#### Implementation

- scale down our query matrix - this is the equivalent of doing $\sqrt{d}$ at the softmax step, we're just doing it earlier on!
- chunk up queries, chunk up the the key and values
- query token i, never attends to key or value tokens `>i` when we're doing causal masking. So what we'll do is only get query tokens to attend to KV token idx `<i`

#### Numerical shorthands

- shorthand for $e^x = 2^{x/ln2}$ for numerical efficiency on hardware

$$
log_2e = \dfrac{log_ee}{log_e2} =  \dfrac{1}{log_e2}
$$

$$
e^x = 2^{xlog_2e} = 2^{\dfrac{x}{log_e2}}
$$


#### Attention masking

Oh boy, flash attention causal masking is no easy feat - here's an attempt at explaining what's happening

- we care about 3 things when causally masking
  - blocks that are above the diagonal (should be masked out)
  - blocks below the diagonal (shouldn't have any mask)
  - where is the diagonal defined??

- where the diagonal is defined: wherever the query token idx = key token idx, and then you just apply the mask above, and no mask below that diagonal!

_Steps to building the mask_

`q_end` = end of the query token chunk  
`kv_end` = end of the key, value token chunk

- **Firstly**: we actually don't even want to perform any operations on the KV vectors that are greater than our query vectors.
  - E.g. query vectors run from 0 to 100, KV vectors run from 0 to 200. Our query vectors will NEVER attend to KV vectors from 101 to 200, so get rid of them!
  - OR in other words, the only KV vectors we care about are the ones that range from 0 -> q_end. And the KV tokens/vectors that will always be safe from masking are the ones before q_start!

- **Secondly**: find me the diagonal
  - it starts where the query chunk starts (token 0 -> 100, token 0 begins the diagonal)
  - it ends where query chunk ends (the last query token: 100, will never attend to anything past it. We don't need any KV vectors past 100 then!)

#### EXAMPLE

Below we have a diagram that is 8 query vectors, attending to 7 key vectors

- 8 query vectors range from 10->17
- 7 query vectors range from 7->13

- First 4 columns (column 7,8,9,10), the query vectors from 10->17 can attend to all of them - denoted by the _blue rectangle_
- HOWEVER...the next 3 columns (column 11,12,13), the query vectors from 11->17 should only attend to some - denoted by the _pink rectangle_
  - `query vector 11`, should only attend to `key vector ..10, 11`
  - `query vector 12`, should only attend to `key vector ..10, 11, 12`
  - `query vector 13`, should only attend to `key vector ..10, 11, 12, 13`
  - `query vector 14`, should only attend to `key vector ..10, 11, 12, 13`
  - `query vector 15`, should only attend to `key vector ..10, 11, 12, 13`


<img src="../assets/flash-5.jpg" width="300">

- notice that the diagonal begins when our key token idx (`kv_start`) = the first query token idx (`q_start`)
- everything before that is not causally masked at all!

#### Creating the causal mask

Use an outer product to compare the query indices with the key indices, whenever the key indices > query indices, MASK! (i.e. set to `False`)

- you'll need to do this for every row, meaning compare the first query index (e.g. 10) with every key index (e.g. 0 -> 64)
- asking is `query idx > key idx`?

Do this using a simple outer product of `(q_chunk, 1) @ (1, k_chunk)`

<img src="../assets/flash-6.jpg" width="300">


In [ ]:
def flash_forward(
	Q:torch.tensor, 
	K:torch.tensor, 
	V:torch.tensor, 
	softmax_scale=None,
	Q_block_size:int=64, 
	KV_block_size:int=64,
	is_causal:bool=False,
):

	batch_size, num_heads, max_seq_len, head_dim = Q.shape
	
	# scale down the query matrix - we won't get a chance to scale down our QK^T matrix since we won't be materializing the full thing, so we can just scale down our query matrix instead

	if softmax_scale is None: 
		softmax_scale = 1.0 / math.sqrt(head_dim)
	
	inv_ln2 = 1/math.log(2) # for softmax, we're going to do 2^(logit) instead of e^(logit)
	
	softmax_scale *= inv_ln2
	# iterate through the batch and num_heads

	Q = Q * softmax_scale

	L = torch.full((batch_size, num_heads, max_seq_len), float('-inf'), dtype=torch.float32) # logsumexp trick
	output = torch.zeros_like(Q) # final output matrix

	for batch in range(batch_size):

		for head in range(num_heads):

			for q_start in range(0, max_seq_len, Q_block_size):

				q_end = min(q_start + Q_block_size, max_seq_len) # make sure we don't go over the amount of tokens we have
				
				# pluck out the query vectors we want
				queries = Q[batch, head, q_start:q_end, :]

				q_indices = torch.arange(q_start, q_end, device=Q.device)

				softmax_max = torch.full((q_end - q_start,), float('-inf'), dtype=torch.float32)
				softmax_sum = torch.zeros((q_end - q_start,), dtype=torch.float32)
				output_block = torch.zeros((q_end - q_start, head_dim), dtype=torch.float32) # final attention output for these query vectors

				if is_causal: 

					# we need to create 2 different masks
					# (1) under the diagonal - no mask
					# (2) on the diagonal - where q_indices > kv_indices, mask
					# (3) above the diagonal - masked out completely

					# easier to understand this code by visualizing tokens 50->150 in the query matrix attending to tokens 100->200 in the key matrix
					kv_start_prediag = 0
					kv_end_prediag = q_start # if our query tokens start from token 50, token 50 should see everything before it, so no masking up until token 50

					kv_start_diag = q_start # now the first part of our diagonal starts - any kv tokens past q_start need to be masked
					kv_end_diag = q_end # any kv tokens past q_end are going to be completely masked, so we should cut off there
					
					kv_ranges = [
						(kv_start_prediag, kv_end_prediag, "prediagonal"),
						(kv_start_diag, kv_end_diag, "diagonal"),
					]
				else:
					kv_ranges = [(0, max_seq_len, 'full')]
				
				for kv_start_range, kv_end_range, pass_type in kv_ranges:

					# first loop: compute the prediagonal in chunks, which occurs from 0 -> kv_start_prediag
					# second loop: compute the diagonal in chunks, creating a mask over parts where q_indices > k_indices
					for kv_start in range(kv_start_range, kv_end_range, KV_block_size):

						# first loop: no masking
						# second loop: masking from q_start in the keys, values matrix through to q_end

						kv_end = min(kv_start+KV_block_size, kv_end_range)

						kv_indices = torch.arange(kv_start, kv_end, device=K.device)

						keys = K[batch, head, kv_start:kv_end, :]
						values = V[batch, head, kv_start:kv_end, :]
						
						# queries attend to these key vectors iteratively
						qk = torch.einsum("qd, kd -> qk", queries, keys)

						if pass_type == 'diagonal':
							causal_mask = q_indices[:, None] >= kv_indices[None, :] # outer product (q_chunk, 1) @ (1, kv_indices)
							qk = qk.masked_fill(~causal_mask, float('-inf')) # mask where indices are true

						####################
						## ONLINE SOFTMAX ##
						####################

						# get the max value (q_end-q_start,) tensor - max value for each query token
						m_new = torch.maximum(qk.max(dim=-1).values, softmax_max) # recall this is just whatever we have from our current chunks
						
						# now our logsumexp trick to get us softmax
						qk = qk - m_new.unsqueeze(-1) # this is the max of each row, subtract it along the rows via broadcasting! (i.e. x - max)
						raw_qk = torch.exp2(qk).to(Q.dtype) # now take 2^all values, getting us the raw exp values (numerator of softmax)
						new_chunk_exp_sum = raw_qk.sum(-1, keepdims=False) # compute the sum of all rows

						# compute the correction factor between prev max, new max!
						alpha = torch.exp2(softmax_max - m_new) # recall all our values have been prev divided by 1/ln2, so we must exponentiate by base of 2

						### LOGSUMEXP value `L`
						softmax_sum = softmax_sum * alpha + new_chunk_exp_sum # new_sum = old_chunks_sum * alpha + new_chunk_exp_sum
						
						#################################
						## UPDATE OUR FINAL ATT VALUES ##
						#################################

						# query tokens q_start->q_end have now attended to kv_start -> kv_end_range, remember we'll do this in a loop to get through all the KV vectors
						# this is just a slice of the KV vectors (KV_block_size)

						# whats saved so far in output_block is 2^x/ln2 for every value - there has been no dividing by the sum, its just the raw exponentiated values right now! 
						# we keep it this way because we don't know the correct max / final exponentiated sum yet for softmax

						output_block = output_block * alpha.unsqueeze(-1) + torch.einsum('qk, kd -> qd', raw_qk, values) # (b, n, q_chunk, k_chunk) @ (b, n, value_chunk, head_dim)

						softmax_max = m_new # set new max

				# add m_i, d_i to our M matrix. This will give us correction factors
				chunk_logsumexp = softmax_max + torch.log2(softmax_sum) # creates our logsumexp trick component = m + log(sum)
				L[batch, head, q_start:q_end] = chunk_logsumexp

				output_block = output_block / softmax_sum.unsqueeze(-1)

				output[batch, head, q_start:q_end, :] = output_block

	return output, L

#### Normalizing the attention output matrix (`output`)

You might notice that we matmul the unnormalized $QK^T$ matrix (`raw_qk`) with the value matrix, which seems problematic but is ok. Some simple reasoning why:

`(3u) @ v = 3(u @ v)` - we can always normalize after the fact, by dividing each row in the attention output by the exponentiated sum per row. This is the exact same whether we did it before matmul with the values matrix or after!


#### Summary of `flash_forward`

- iterate through `batch_size` and `num_heads`, doing one batch / head i of each batch, at a time

- grab a chunk of the query vectors (`Q_block_size`)
  - iteratively grab chunks of the key vectors (`KV_block_size`), and matmul them with the query vector chunk

- NOTE: we'll need to do some causal masking
  1.  all KV vectors before `q_start` won't need any masking at all, do full attention
  2.  all KV vectors past `q_start` will need to be masked - this mask will exist whenver `q_index > k_index`, intuitively query token 10 never attends to key token 11

- After we computed our raw qk logits - exponentiate them to get our `raw_qk` matrix, and keep a running tally of the max value of each row of qk `softmax_max`, and also an adjusted sum of each row `softmax_sum`

- As we go matmul with the values vector `V` to get our unnormalized attention output. And every iteration, correct the previous values by multiplying by `alpha`, our correction value as we update the max of each row

- Now once we've completed a row of qk (i.e. query token `i` has attended to every key token), we now have the exponentiated sum of the row. Divide the row by this to get the true softmax (normalized) attention output!


In [37]:
import torch.nn.functional as F 

q = torch.randn(2,4,1024,64)
k = torch.randn(2,4,1024,64)
v = torch.randn(2,4,1024,64)

out, L = flash_forward(q,k,v, is_causal=True)
ref = F.scaled_dot_product_attention(q,k,v, is_causal=True)

print(f"Do our methods match? Answer: ",torch.allclose(out, ref, atol=1e-6))
print(f"Largest point difference: ",(out-ref).abs().max().item())

Do our methods match? Answer:  True
Largest point difference:  1.6689300537109375e-06
